# 01 · Descarga de datos SINIM

**Objetivo:** descargar las variables definidas en `src/config.py`, armar los paneles y verificar que los datos estén completos.

**Genera:**
- `data/raw/<grupo>.parquet`: un archivo por grupo, tal cual lo entrega SINIM.
- `data/processed/panel_largo.parquet`: todas las variables, una fila por comuna-año-variable.
- `data/processed/panel_ancho.parquet`: una fila por comuna-año, una columna por variable.

**Uso:** ejecutar con `FORZAR = True` solo cuando cambien las variables, las comunas o los años en `config.py`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import pandas as pd
from src.config import COMMUNES, DATA_PROCESSED, GROUPS, YEARS
from src.sinim import procesar

FORZAR = False

In [2]:
## 1. Ejecutar el pipeline

ancho = procesar(forzar=FORZAR)
largo = pd.read_parquet(DATA_PROCESSED / "panel_largo.parquet")

presupuesto - parquet guardado ✅
ingresos - parquet guardado ✅
gastos_areas - parquet guardado ✅
personal - parquet guardado ✅
servicios - parquet guardado ✅
transferencias - parquet guardado ✅
indicadores - parquet guardado ✅
contexto - parquet guardado ✅
gastos_subtitulos - parquet guardado ✅


## 2. Verificaciones

Cada grupo debería tener (n.º de variables × n.º de años × n.º de comunas) filas.

In [3]:
esperado = {nombre: len(vars) * len(YEARS) * len(COMMUNES) for nombre, vars in GROUPS.items()}

revision = pd.DataFrame({
    "esperado": pd.Series(esperado),
    "obtenido": largo["grupo"].value_counts(),
})
revision["ok"] = revision["esperado"] == revision["obtenido"]
revision

,esperado,obtenido,ok
contexto,44,44,True
gastos_areas,66,66,True
gastos_subtitulos,44,44,True
indicadores,99,99,True
ingresos,132,132,True
personal,55,55,True
presupuesto,77,77,True
servicios,55,55,True
transferencias,55,55,True


In [4]:
print("Forma del panel ancho:", ancho.shape)
print("Variables distintas:", largo["variable"].nunique())
print("Alias faltantes:", largo["variable"].isna().sum())
print("Valores vacíos:", largo["value"].isna().sum())

Forma del panel ancho: (11, 60)
Variables distintas: 57
Alias faltantes: 0
Valores vacíos: 10


In [ ]:
## Informacion de años faltantes

largo[largo["value"].isna()].groupby("variable")["anio"].apply(list)

variable
pct_poblacion_rural                            [2017, 2016, 2015]
viviendas              [2021, 2020, 2019, 2018, 2017, 2016, 2015]
Name: anio, dtype: object